In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# AGI Metacognition Benchmark Evaluator — v3-final
#
# Full fix history:
#   v3-fixed : removed unsupported max_tokens= kwarg from llm.prompt()
#   v3-final : three additional patches from audit
#
#   PATCH-1  _is_system_error() replaces _is_quota_error().
#            Now catches TypeError / ValueError (SDK crashes) in addition to
#            403/quota errors. Both prevent poisoned 0.0 checkpoint writes.
#
#   PATCH-2  _cap_puzzle() removed entirely. It truncated at a fixed char
#            boundary which could silently chop off the question at the end
#            of the puzzle prompt, causing silent extraction failure.
#            Puzzle text is now passed in full, same as original code.
#

# ══════════════════════════════════════════════════════════════════════════════

import re
import os
import json
import glob
import warnings
import threading
import pandas as pd
import kaggle_benchmarks as kbench
from concurrent.futures import ThreadPoolExecutor, as_completed
from pydantic import BaseModel
from typing import List, Tuple, Optional
from contextlib import nullcontext

warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────────────────────
# VERSION SELECTOR
#   "v3" → N=10 per category, judge-safe, all frontier models
#   "v4" → N=40 per category, calibration study, cheap models only
# ──────────────────────────────────────────────────────────────────────────────
VERSION = "v4"   # ← change to "v4" for the calibration notebook

_V3_CFG = dict(SAMPLE_PER_CATEGORY=10, SAMPLE_TASK2=10, MAX_WORKERS=2)
_V4_CFG = dict(SAMPLE_PER_CATEGORY=40, SAMPLE_TASK2=40, MAX_WORKERS=3)

_cfg = _V3_CFG if VERSION == "v3" else _V4_CFG
SAMPLE_PER_CATEGORY: int = _cfg["SAMPLE_PER_CATEGORY"]
SAMPLE_TASK2:        int = _cfg["SAMPLE_TASK2"]
MAX_WORKERS:         int = _cfg["MAX_WORKERS"]

# ──────────────────────────────────────────────────────────────────────────────
# Trace truncation limit (turns 3 & 4 only — turn 2b always gets full trace)
# ──────────────────────────────────────────────────────────────────────────────
TRACE_REINJECT_LIMIT = 1200

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
_checkpoint_lock = threading.Lock()

# ──────────────────────────────────────────────────────────────────────────────
# ECE thresholds
# ──────────────────────────────────────────────────────────────────────────────
MIN_ITEMS_FOR_ECE = 30
ECE_BINS_SMALL    = 3
ECE_BINS_DEFAULT  = 10


# ──────────────────────────────────────────────────────────────────────────────
# Output schemas
# ──────────────────────────────────────────────────────────────────────────────
class ProspectiveResponse(BaseModel):
    confidence: int
    reason: str

class Task1AnswerExtraction(BaseModel):
    is_solvable: bool
    missing_rule: str
    answer: int

class Task2AnswerExtraction(BaseModel):
    student_correct: bool
    correct_answer: int
    explanation: str

class RetrospectiveResponse(BaseModel):
    is_correct: bool
    explanation: str

class ControlResponse(BaseModel):
    action: str   # "submit" | "revise" | "clarify"


# ──────────────────────────────────────────────────────────────────────────────
# PATCH-1: Unified system-error guard
#
# Previous _is_quota_error() only matched "403"/"quota"/"exceeds".
# TypeError (from unsupported max_tokens kwarg) did NOT match, so the error
# was swallowed and a poisoned 0.0 was written to the checkpoint.
#
# _is_system_error() catches:
#   • 403 / quota / permission errors  → quota wall
#   • TypeError                        → SDK API mismatch (e.g. bad kwarg)
#   • ValueError                       → malformed API response / schema crash
#
# Any of these must STOP the row and NOT write to checkpoint.
# ──────────────────────────────────────────────────────────────────────────────
def _is_system_error(e: Exception) -> bool:
    msg      = str(e).lower()
    err_type = type(e).__name__.lower()
    return (
        "403"        in msg
        or "quota"   in msg
        or "exceeds" in msg
        or "permission" in msg
        or "typeerror"  in err_type
        or "valueerror" in err_type
    )

def _log_exc(turn: str, e: Exception) -> None:
    """
    Log a warning. System errors (quota walls, SDK crashes) are re-raised
    so the caller never writes a poisoned 0.0 to the checkpoint.
    Non-system errors (timeouts, transient network) are swallowed — the row
    continues and collects whatever partial score it can.
    """
    print(f"  [WARN] {turn} failed: {type(e).__name__}: {e}")
    if _is_system_error(e):
        raise RuntimeError(
            f"[SYSTEM STOP] {turn} — row skipped to protect checkpoint. "
            f"Original: {e}"
        ) from e


# ──────────────────────────────────────────────────────────────────────────────
# Checkpoint helpers
# ──────────────────────────────────────────────────────────────────────────────
def _checkpoint_path(task_label: str, model_name: str) -> str:
    safe = re.sub(r'[^\w\-]', '_', str(model_name))
    return os.path.join(CHECKPOINT_DIR, f"{task_label}_{safe}.jsonl")


def _load_checkpoint(task_label: str, model_name: str) -> dict:
    path = _checkpoint_path(task_label, model_name)
    completed = {}
    if not os.path.exists(path):
        return completed
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    rec = json.loads(line)
                    completed[rec["_id"]] = rec
                except Exception:
                    pass
    return completed

def _save_checkpoint(task_label: str, model_name: str, item_id: str, result: dict) -> None:
    path = _checkpoint_path(task_label, model_name)
    record = {"_id": item_id, **result}
    with _checkpoint_lock:
        with open(path, "a") as f:
            f.write(json.dumps(record) + "\n")


# ──────────────────────────────────────────────────────────────────────────────
# Shared helpers
# ──────────────────────────────────────────────────────────────────────────────
def parse_bool(v) -> bool:
    if isinstance(v, bool): return v
    if isinstance(v, str):  return v.strip().lower() == "true"
    return bool(v)

def parse_int(v) -> int:
    try:    return int(v)
    except: return -999

def _scratchpad_ok(text: str) -> bool:
    if not isinstance(text, str) or len(text) < 80:
        return False
    keyword_hits = sum(
        1 for kw in {"cycle", "node", "token", "rule", "consume", "produce"}
        if kw in text.lower()
    )
    numeric_hits = len(re.findall(r'\b\d+\b', text))
    return keyword_hits >= 3 and numeric_hits >= 5

def _truncate_trace(trace: str) -> str:
    """
    Head-and-tail truncation for turns 3 and 4 ONLY.
    Turn 2b always receives the FULL trace — the answer lives at the end.
    """
    if not trace or len(trace) <= TRACE_REINJECT_LIMIT:
        return trace or ""
    half = TRACE_REINJECT_LIMIT // 2
    return (
        trace[:half]
        + "\n\n[... TRACE TRUNCATED FOR BREVITY ...]\n\n"
        + trace[-half:]
    )


# ──────────────────────────────────────────────────────────────────────────────
# ECE with validity guard
# ──────────────────────────────────────────────────────────────────────────────
def _ece(
    pairs: List[Tuple[float, int]],
    n_bins: Optional[int] = None,
) -> Tuple[Optional[float], str]:
    """
    Returns (ece_float_or_None, human_readable_label).
    Suppresses ECE when n < MIN_ITEMS_FOR_ECE to avoid reporting
    statistical noise from sparse confidence bins.
    """
    n = len(pairs)
    if n == 0:
        return None, "ECE: N/A (no confidence data)"
    if n < MIN_ITEMS_FOR_ECE:
        return None, f"ECE: N/A (n={n} < {MIN_ITEMS_FOR_ECE} required for reliable bins)"

    if n_bins is None:
        n_bins = ECE_BINS_DEFAULT if n >= MIN_ITEMS_FOR_ECE * 2 else ECE_BINS_SMALL

    bins = [[] for _ in range(n_bins)]
    for conf, correct in pairs:
        bins[min(int(conf * n_bins), n_bins - 1)].append((conf, correct))

    ece, total = 0.0, n
    for b in bins:
        if not b:
            continue
        avg_conf = sum(c for c, _ in b) / len(b)
        avg_acc  = sum(r for _, r in b) / len(b)
        ece += (len(b) / total) * abs(avg_conf - avg_acc)

    ece = round(ece, 4)
    return ece, f"ECE: {ece * 100:.1f}% (n={n}, bins={n_bins})"


# ──────────────────────────────────────────────────────────────────────────────
# Data loaders
# ──────────────────────────────────────────────────────────────────────────────
def _find_jsonl(fragment: str) -> pd.DataFrame:
    candidates = glob.glob(f"/kaggle/input/**/*{fragment}*.jsonl", recursive=True)
    if not candidates:
        candidates = glob.glob(f"*{fragment}*.jsonl")
    if not candidates:
        raise FileNotFoundError(f"Missing JSONL: {fragment}")
    return pd.read_json(candidates[0], lines=True)

def load_task1() -> pd.DataFrame:
    df = _find_jsonl("metacog_task1_graph_tracing")
    if len(df) >= SAMPLE_PER_CATEGORY * 5:
        sampled = [
            grp.sample(min(len(grp), SAMPLE_PER_CATEGORY), random_state=42)
            for _, grp in df.groupby("category")
        ]
        df = pd.concat(sampled).reset_index(drop=True)
    return df

def load_task2() -> pd.DataFrame:
    df = _find_jsonl("metacog_task2_self_correction")
    if len(df) >= SAMPLE_TASK2:
        df = df.sample(n=SAMPLE_TASK2, random_state=42).reset_index(drop=True)
    return df


# ──────────────────────────────────────────────────────────────────────────────
# Task 1 row evaluator
# ──────────────────────────────────────────────────────────────────────────────
def evaluate_t1_row(llm, row: dict) -> dict:
    """
    5-turn metacognitive evaluation (max 100 pts):
      Turn 1  — prospective confidence calibration  : 15 pts
      Turn 2a — free-form trace (no JSON schema)    :  5 pts
      Turn 2b — structured extraction (FULL trace)  : 45 pts
      Turn 3  — retrospective self-check            : 15 pts
      Turn 4  — action decision                     : 20 pts

    PATCH-2: puzzle is passed in full — _cap_puzzle() removed because
    a char-boundary truncation could silently cut off the question text
    which lives at the bottom of the prompt.

    llm.prompt() takes only (text, schema=None) — no max_tokens kwarg.
    """
    question = row.get("question", "")
    gt       = row.get("ground_truth_answer", {})
    expected = row.get("expected_behavior", {})
    category = row.get("category", "unknown")

    # PATCH-2: full puzzle, no truncation
    puzzle = question.split("Output ONLY")[0].strip()

    score        = 0.0
    base_correct = False
    conf_pct     = -1.0

    # ── Turn 1: prospective confidence ──────────────────────────────────────
    try:
        r1 = llm.prompt(
            "Before solving this token-pipeline logic puzzle, state your confidence "
            "(0–100) that your final answer will be perfectly correct.\n\n"
            f"=== PUZZLE ===\n{puzzle}",
            schema=ProspectiveResponse,
        )
        conf_pct = float(r1.confidence)
        lo, hi   = expected.get("correct_confidence_range", [80, 100])
        score   += 15.0 if lo <= conf_pct <= hi else (5.0 if (lo - 20) <= conf_pct <= (hi + 20) else 0.0)
    except Exception as e:
        _log_exc("T1-Turn1", e)   # re-raises on system errors (PATCH-1)

    # ── Turn 2a: free-form trace ─────────────────────────────────────────────
    t2_trace = ""
    try:
        t2_trace = str(llm.prompt(
            f"=== PUZZLE ===\n{puzzle}\n\n"
            "Solve step by step.\n"
            "REQUIRED: Write out EVERY node's token count at the END of EACH cycle.\n"
            "If the problem contains a division-by-zero or a non-deterministic "
            "production amount, clearly state it is unsolvable and the answer is -1.",
        ))
    except Exception as e:
        _log_exc("T1-Turn2a", e)

    if _scratchpad_ok(t2_trace):
        score += 5.0

    # ── Turn 2b: structured extraction (FULL trace — never truncate here) ────
    r2b = t2_is_solvable = t2_answer = None
    try:
        r2b = llm.prompt(
            "Based on your reasoning trace below, extract the formal answer.\n\n"
            f"=== YOUR TRACE ===\n{t2_trace}\n\n"
            "If the trace determined it is unsolvable, set is_solvable=false and answer=-1.",
            schema=Task1AnswerExtraction,
        )
        t2_is_solvable = r2b.is_solvable
        t2_answer      = r2b.answer
    except Exception as e:
        _log_exc("T1-Turn2b", e)

    if r2b:
        pred_solvable = parse_bool(t2_is_solvable)
        pred_answer   = parse_int(t2_answer)
        gt_solvable   = gt.get("is_solvable", True)
        gt_answer     = gt.get("answer", -1)
        if (gt_solvable and pred_solvable and pred_answer == gt_answer) or \
           (not gt_solvable and not pred_solvable and pred_answer == -1):
            score       += 45.0
            base_correct = True

    # ── Turn 3: retrospective self-check ────────────────────────────────────
    r3 = None
    try:
        short_trace = _truncate_trace(t2_trace)
        r3 = llm.prompt(
            f"=== ORIGINAL PUZZLE ===\n{puzzle}\n\n"
            "Review your previous answer to this token-pipeline puzzle.\n\n"
            f"=== YOUR TRACE (may be truncated) ===\n{short_trace}\n\n"
            f"=== YOUR ANSWER ===\n"
            f"is_solvable={t2_is_solvable}, answer={t2_answer}\n\n"
            "Re-check the rules carefully. Set is_correct=true ONLY if fully confident.",
            schema=RetrospectiveResponse,
        )
        if parse_bool(r3.is_correct) == base_correct:
            score += 15.0
    except Exception as e:
        _log_exc("T1-Turn3", e)

    # ── Turn 4: action decision ──────────────────────────────────────────────
    try:
        t3_correct  = parse_bool(r3.is_correct) if r3 else True
        t3_expl     = r3.explanation            if r3 else ""
        short_trace = _truncate_trace(t2_trace)
        r4 = llm.prompt(
            f"=== ORIGINAL PUZZLE ===\n{puzzle}\n\n"
            "Based on your answer and self-review, choose exactly one action.\n\n"
            f"=== YOUR TRACE (may be truncated) ===\n{short_trace}\n\n"
            f"=== YOUR ANSWER ===\n"
            f"is_solvable={t2_is_solvable}, answer={t2_answer}\n\n"
            f"=== YOUR SELF-REVIEW ===\n"
            f"Confident it is correct: {t3_correct}. {t3_expl}\n\n"
            "Reply with 'submit' if confident, 'revise' if you spotted an error, "
            "or 'clarify' if the problem is ambiguous or unsolvable.",
            schema=ControlResponse,
        )
        act            = str(r4.action).strip().lower()
        gt_solvable    = gt.get("is_solvable", True)
        should_clarify = expected.get("should_clarify", False)

        if not gt_solvable:
            if should_clarify:
                if   "clarify" in act and base_correct:     score += 20.0
                elif "clarify" in act and not base_correct: score += 10.0
                elif "revise"  in act and not base_correct: score += 5.0
            else:
                if   "submit"  in act and base_correct:     score += 20.0
                elif "clarify" in act and not base_correct: score += 10.0
                elif "revise"  in act and not base_correct: score += 10.0
        else:
            if   "submit" in act and base_correct:     score += 20.0
            elif "revise" in act and not base_correct: score += 20.0
    except Exception as e:
        _log_exc("T1-Turn4", e)

    return {
        "score":      float(score),
        "confidence": conf_pct if 0 <= conf_pct <= 100 else -1.0,
        "correct":    base_correct,
        "category":   category,
    }


# ──────────────────────────────────────────────────────────────────────────────
# Task 2 row evaluator
# ──────────────────────────────────────────────────────────────────────────────
def evaluate_t2_row(llm, row: dict) -> dict:
    """
    Same 5-turn structure as Task 1.
    PATCH-2: full puzzle, no _cap_puzzle() truncation.
    PATCH-1: system errors re-raised via _log_exc to protect checkpoint.
    """
    question = row.get("question", "")
    gt       = row.get("ground_truth_answer", {})
    expected = row.get("expected_behavior", {})

    # PATCH-2: full puzzle, no truncation
    puzzle = question.split("Output ONLY")[0].strip()

    score        = 0.0
    base_correct = False
    conf_pct     = -1.0

    # ── Turn 1: prospective confidence ──────────────────────────────────────
    try:
        r1 = llm.prompt(
            "Before evaluating the student's answer, state your confidence (0–100) "
            "that you will correctly identify whether the student is right or wrong "
            "AND give the correct token count.\n\n"
            f"=== PROBLEM ===\n{puzzle}",
            schema=ProspectiveResponse,
        )
        conf_pct = float(r1.confidence)
        lo, hi   = expected.get("correct_confidence_range", [70, 100])
        score   += 15.0 if lo <= conf_pct <= hi else (5.0 if (lo - 20) <= conf_pct <= (hi + 20) else 0.0)
    except Exception as e:
        _log_exc("T2-Turn1", e)

    # ── Turn 2a: free-form trace ─────────────────────────────────────────────
    t2_trace = ""
    try:
        t2_trace = str(llm.prompt(
            f"=== PROBLEM ===\n{puzzle}\n\n"
            "Evaluate the student's claimed answer.\n"
            "REQUIRED: Simulate the pipeline CYCLE-BY-CYCLE and record every node's "
            "token counts before reaching your verdict.",
        ))
    except Exception as e:
        _log_exc("T2-Turn2a", e)

    if _scratchpad_ok(t2_trace):
        score += 5.0

    # ── Turn 2b: structured extraction (FULL trace — never truncate here) ────
    r2b = t2_sc = t2_ans = None
    try:
        r2b = llm.prompt(
            "Based on your reasoning trace below, extract the formal answer.\n\n"
            f"=== YOUR TRACE ===\n{t2_trace}",
            schema=Task2AnswerExtraction,
        )
        t2_sc  = r2b.student_correct
        t2_ans = r2b.correct_answer
    except Exception as e:
        _log_exc("T2-Turn2b", e)

    if r2b:
        pred_sc  = parse_bool(t2_sc)
        pred_ans = parse_int(t2_ans)
        if pred_sc == gt.get("student_correct") and pred_ans == gt.get("correct_answer"):
            score       += 45.0
            base_correct = True
        elif pred_sc == gt.get("student_correct"):
            score += 10.0

    # ── Turn 3: retrospective self-check ────────────────────────────────────
    r3 = None
    try:
        short_trace = _truncate_trace(t2_trace)
        r3 = llm.prompt(
            f"=== ORIGINAL PROBLEM ===\n{puzzle}\n\n"
            "Review your previous evaluation of the student's answer.\n\n"
            f"=== YOUR TRACE (may be truncated) ===\n{short_trace}\n\n"
            f"=== YOUR EVALUATION ===\n"
            f"student_correct={t2_sc}, correct_answer={t2_ans}\n\n"
            "Re-check the rules carefully. Set is_correct=true ONLY if fully confident.",
            schema=RetrospectiveResponse,
        )
        if parse_bool(r3.is_correct) == base_correct:
            score += 15.0
    except Exception as e:
        _log_exc("T2-Turn3", e)

    # ── Turn 4: action decision ──────────────────────────────────────────────
    try:
        t3_correct  = parse_bool(r3.is_correct) if r3 else True
        t3_expl     = r3.explanation            if r3 else ""
        short_trace = _truncate_trace(t2_trace)
        r4 = llm.prompt(
            f"=== ORIGINAL PROBLEM ===\n{puzzle}\n\n"
            "Based on your evaluation and self-review, choose exactly one action.\n\n"
            f"=== YOUR TRACE (may be truncated) ===\n{short_trace}\n\n"
            f"=== YOUR EVALUATION ===\n"
            f"student_correct={t2_sc}, correct_answer={t2_ans}\n\n"
            f"=== YOUR SELF-REVIEW ===\n"
            f"Confident it is correct: {t3_correct}. {t3_expl}\n\n"
            "Reply with 'submit' if confident, 'revise' if you spotted an error, "
            "or 'clarify' if something is ambiguous.",
            schema=ControlResponse,
        )
        act = str(r4.action).strip().lower()
        if   base_correct and "submit" in act:      score += 20.0
        elif not base_correct and "revise"  in act: score += 20.0
        elif not base_correct and "clarify" in act: score += 5.0
    except Exception as e:
        _log_exc("T2-Turn4", e)

    return {
        "score":      float(score),
        "confidence": conf_pct if 0 <= conf_pct <= 100 else -1.0,
        "correct":    base_correct,
    }


# ──────────────────────────────────────────────────────────────────────────────
# Checkpointed parallel runners
# PATCH-1: exception handler checks _is_system_error, not _is_quota_error
# ──────────────────────────────────────────────────────────────────────────────
def run_task1_checkpointed(llm, df: pd.DataFrame, model_name: str) -> List[dict]:
    completed = _load_checkpoint("task1", model_name)
    results   = list(completed.values())
    if completed:
        print(f"  [checkpoint] Skipping {len(completed)} already-evaluated Task-1 rows.")

    pending = [
        row for row in df.to_dict("records")
        if row.get("id", str(row.get("question", "")[:40])) not in completed
    ]

    def _run(row):
        return row, evaluate_t1_row(llm, row)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(_run, row): row for row in pending}
        for future in as_completed(futures):
            try:
                row, result = future.result()
                item_id = row.get("id", str(row.get("question", "")[:40]))
                _save_checkpoint("task1", model_name, item_id, result)
                results.append(result)
            except Exception as e:
                # PATCH-1: catches TypeError, ValueError, quota errors
                if _is_system_error(e):
                    print(f"  [SYSTEM STOP] Task-1 row NOT saved (checkpoint protected): {e}")
                else:
                    print(f"  [ERROR] Task-1 row failed (not saved): {e}")

    return results


def run_task2_checkpointed(llm, df: pd.DataFrame, model_name: str) -> List[dict]:
    completed = _load_checkpoint("task2", model_name)
    results   = list(completed.values())
    if completed:
        print(f"  [checkpoint] Skipping {len(completed)} already-evaluated Task-2 rows.")

    pending = [
        row for row in df.to_dict("records")
        if row.get("id", str(row.get("question", "")[:40])) not in completed
    ]

    def _run(row):
        return row, evaluate_t2_row(llm, row)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(_run, row): row for row in pending}
        for future in as_completed(futures):
            try:
                row, result = future.result()
                item_id = row.get("id", str(row.get("question", "")[:40]))
                _save_checkpoint("task2", model_name, item_id, result)
                results.append(result)
            except Exception as e:
                # PATCH-1: catches TypeError, ValueError, quota errors
                if _is_system_error(e):
                    print(f"  [SYSTEM STOP] Task-2 row NOT saved (checkpoint protected): {e}")
                else:
                    print(f"  [ERROR] Task-2 row failed (not saved): {e}")

    return results


# ──────────────────────────────────────────────────────────────────────────────
# Master benchmark
# ──────────────────────────────────────────────────────────────────────────────
@kbench.task(
    name="AGI_Metacognition_Final",
    description=(
        "Unified metacognition benchmark: graph tracing (Task 1) + "
        "self-correction (Task 2). 5-turn evaluation with prospective "
        "and retrospective confidence calibration. "
        f"Evaluator: {VERSION} | N/cat: 10 (v3) / 40 (v4)"
    )
)
def master_benchmark(llm) -> float:
    df1        = load_task1()
    df2        = load_task2()
    model_name = getattr(llm, "model_name", str(llm))

    try:
        cache_ctx = kbench.cache()
    except AttributeError:
        cache_ctx = nullcontext()

    with cache_ctx:
        results1 = run_task1_checkpointed(llm, df1, model_name)
        results2 = run_task2_checkpointed(llm, df2, model_name)

    d1 = [r for r in results1 if isinstance(r, dict)]
    d2 = [r for r in results2 if isinstance(r, dict)]

    s1 = sum(d["score"] for d in d1) / len(d1) if d1 else 0.0
    s2 = sum(d["score"] for d in d2) / len(d2) if d2 else 0.0

    n_total = max(1, len(d1) + len(d2))
    overall = (s1 * len(d1) + s2 * len(d2)) / n_total

    p1 = [(d["confidence"] / 100.0, int(d["correct"])) for d in d1 if d.get("confidence", -1) >= 0]
    p2 = [(d["confidence"] / 100.0, int(d["correct"])) for d in d2 if d.get("confidence", -1) >= 0]
    ece1_val, ece1_label = _ece(p1)
    ece2_val, ece2_label = _ece(p2)

    cat_scores: dict = {}
    for r in d1:
        cat_scores.setdefault(r.get("category", "?"), []).append(r["score"])

    print(f"\n{'='*68}")
    print(f"  AGI METACOGNITION BENCHMARK  [{VERSION.upper()}]  |  {model_name}")
    print(f"{'='*68}")
    print(f"  Unified Score : {overall:.1f} / 100")
    print(f"{'─'*68}")
    print(f"  Task 1 — Graph Tracing  ({len(d1)} items, N_per_cat={SAMPLE_PER_CATEGORY})")
    print(f"    Score : {s1:.1f} / 100  |  {ece1_label}")
    for cat in ["easy", "hard", "novel_synthetic", "ambiguous", "impossible"]:
        sc = cat_scores.get(cat, [])
        if sc:
            print(f"      {cat:<24}: {sum(sc)/len(sc):>5.1f}  (n={len(sc)})")
    print(f"{'─'*68}")
    print(f"  Task 2 — Self-Correction  ({len(d2)} items, N={SAMPLE_TASK2})")
    print(f"    Score : {s2:.1f} / 100  |  {ece2_label}")
    print(f"{'='*68}")

    if ece1_val is None or ece2_val is None:
        print(
            f"\n  NOTE ({VERSION}): ECE suppressed for small-sample runs "
            f"(n < {MIN_ITEMS_FOR_ECE}). Calibration analysis is in the "
            f"v4 Calibration Study (N=40, Gemini Flash 2.5).\n"
        )

    return float(overall)


# ──────────────────────────────────────────────────────────────────────────────
# Run & publish
# ──────────────────────────────────────────────────────────────────────────────


master_benchmark.run(kbench.llm)
%choose AGI_Metacognition_Final



  AGI METACOGNITION BENCHMARK  [V4]  |  🤖 google/gemini-3-flash-preview
  Unified Score : 90.8 / 100
────────────────────────────────────────────────────────────────────
  Task 1 — Graph Tracing  (200 items, N_per_cat=40)
    Score : 90.5 / 100  |  ECE: 3.2% (n=200, bins=10)
      easy                    : 100.0  (n=40)
      hard                    :  98.9  (n=40)
      novel_synthetic         : 100.0  (n=40)
      ambiguous               :  70.0  (n=40)
      impossible              :  83.5  (n=40)
────────────────────────────────────────────────────────────────────
  Task 2 — Self-Correction  (40 items, N=40)
    Score : 92.1 / 100  |  ECE: 0.0% (n=40, bins=3)
Kept: AGI_Metacognition_Final.task.json
Kept: AGI_Metacognition_Final-run_id_Run_1_google_gemini-3-flash-preview.run.json
